# 05 · Señales y riesgo

**Objetivo:** convertir los indicadores del notebook 04 en **señales de compra y de venta** con reglas explícitas, y calcular el riesgo que acompaña a cada señal: stops, volatilidad, Sharpe y estacionalidad.

> Las señales son **técnicas e informativas**, no una recomendación de inversión.

**Entradas:**
- `data/cache/precios_historicos.parquet` (notebook 02): precios diarios ajustados, 10 años, 1.000 empresas
- `data/cache/indicadores.parquet` (notebook 04): los indicadores por empresa y día

**Salidas** (las usan el backtesting del notebook 09 y el screener de `app.py`):
- `data/cache/senales.parquet`: una fila por empresa y día, con el régimen, la liquidez, cada mecanismo y la señal final
- `data/cache/riesgo_empresas.parquet`: volatilidad, Sharpe y ATR relativo del último año, por empresa
- `data/cache/estacionalidad_mensual.parquet` y `estacionalidad_semanal.parquet`: estacionalidad de los últimos 5 años, por empresa

In [1]:
import time

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

from funciones import (
    ADX_UMBRAL_TENDENCIA,
    CARPETA_CACHE,
    SESIONES_CALENTAMIENTO,
    VOLUMEN_MINIMO_DOLARES,
    calcular_estacionalidad,
    calcular_stops_operacion,
    calcular_volatilidad_y_sharpe,
    detectar_senales,
)

load_dotenv();

## Glosario

| Indicador | En una línea | Uso aquí |
|---|---|---|
| **RSI** | Mide de 0 a 100 si el precio ha subido o bajado demasiado rápido (< 30 sobrevendido, > 70 sobrecomprado) | Señal en régimen lateral |
| **MACD** | Diferencia entre dos medias móviles: indica el impulso de la tendencia | Solo visual (interfaz) |
| **ADX** | Mide la **fuerza** de la tendencia, sin dirección (≥ 25 = hay tendencia) | Decide el régimen |
| **ATR** | Rango medio diario del precio en dólares: cuánto se mueve normalmente en un día | Distancia de los stops |
| **CMF** | Flujo de dinero de −1 a +1: si el volumen acompaña a las subidas o a las bajadas | Confirma el cruce de EMA |
| **EMA** | Media móvil exponencial: da más peso a los días recientes. Se cruzan la de 4 y la de 8 sesiones | Señal en régimen de tendencia |
| **Donchian** | Máximo y mínimo de las últimas 20 sesiones | Señal de ruptura |
| **Bollinger** | Media de 20 sesiones ± 2 desviaciones típicas: el rango "normal" del precio | Rango esperado (screener) |
| **Fibonacci** | Niveles intermedios (23,6 %… 78,6 %) entre el máximo y el mínimo de 45 sesiones | Rango esperado (screener) |

## 1. ¿De qué datos partimos?

Se unen los precios y los indicadores por fecha y empresa.

In [2]:
precios = pd.read_parquet(CARPETA_CACHE / "precios_historicos.parquet")
indicadores = pd.read_parquet(CARPETA_CACHE / "indicadores.parquet")
datos = precios.merge(indicadores, on=["fecha", "ticker"], how="inner", validate="one_to_one")
datos = datos.sort_values(["ticker", "fecha"]).reset_index(drop=True)

# Número de sesión dentro del histórico de cada empresa (0 = su primer día)
datos["numero_sesion"] = datos.groupby("ticker").cumcount()

print("Filas:", len(datos), "· Empresas:", datos["ticker"].nunique())
print("Fechas:", datos["fecha"].min().date(), "→", datos["fecha"].max().date())

Filas: 2281091 · Empresas: 1000
Fechas: 2016-10-10 → 2026-10-09


**Conclusión:** 1.000 empresas con unos 10 años de datos diarios cada una.

## 2. ¿Se puede uno fiar del ADX desde el primer día?

Cada día se clasifica con el ADX: **tendencia** si ADX ≥ 25 y **lateral** si es menor. El problema es que el ADX sale inflado en las primeras sesiones de cada empresa, hasta que acumula historia. El gráfico compara el % de empresas en tendencia cada mes, **sin** descartar nada y **descartando las primeras 60 sesiones** de cada empresa (calentamiento).

In [3]:
datos["en_tendencia"] = datos["adx"] >= ADX_UMBRAL_TENDENCIA
con_adx = datos[datos["adx"].notna()]
pasado_calentamiento = con_adx[con_adx["numero_sesion"] >= SESIONES_CALENTAMIENTO]

sin_calentamiento = con_adx.groupby(pd.Grouper(key="fecha", freq="ME"))["en_tendencia"].mean() * 100
con_calentamiento = pasado_calentamiento.groupby(pd.Grouper(key="fecha", freq="ME"))["en_tendencia"].mean() * 100

grafico = go.Figure()
grafico.add_trace(go.Scatter(x=sin_calentamiento.index, y=sin_calentamiento, name="sin calentamiento"))
grafico.add_trace(go.Scatter(x=con_calentamiento.index, y=con_calentamiento, name="con calentamiento de 60 sesiones"))
grafico.update_layout(title="% de empresas en régimen de tendencia (ADX ≥ 25), por mes",
                      xaxis_title="Mes", yaxis_title="% de empresas-día en tendencia")
grafico.show(renderer="notebook_connected")

print(f"Con calentamiento: {pasado_calentamiento['en_tendencia'].mean():.0%} del tiempo en tendencia")

Con calentamiento: 38% del tiempo en tendencia


Sin calentamiento, octubre y noviembre de 2016 (el arranque del histórico) salen con más del 80 % de las empresas en tendencia, casi como el desplome de la COVID en marzo de 2020 (86 %), sin que pasara nada parecido en el mercado. Es un artefacto del cálculo: con el calentamiento, enero de 2017 baja al 33 % y el único pico es **marzo de 2020**.

**Conclusión:** las primeras **60 sesiones** de cada empresa no tienen régimen ni señales.

## 3. ¿Qué días son demasiado poco líquidos para operar?

Algunas empresas apenas se negociaban en EEUU al principio de su histórico. Una señal en esos días no se podría ejecutar. Se mide el **volumen en dólares** (cierre × volumen), con la mediana de las últimas 20 sesiones, y se mira cuánto quedaría fuera con distintos mínimos:

In [4]:
datos["volumen_dolares"] = datos["close"] * datos["volume"]
datos["volumen_dolares_mediano"] = datos.groupby("ticker")["volumen_dolares"].transform(
    lambda serie: serie.rolling(20).median())

filas = []
for minimo in [100_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]:
    por_debajo = datos["volumen_dolares_mediano"] < minimo
    filas.append({
        "mínimo (USD/día)": minimo,
        "% de filas por debajo": round(100 * por_debajo.mean(), 2),
        "empresas afectadas en algún momento": datos.loc[por_debajo, "ticker"].nunique(),
    })
pd.DataFrame(filas)

,mínimo (USD/día),% de filas por debajo,empresas afectadas en algún momento
0,100000,0.75,39
1,1000000,1.60,85
2,2000000,2.18,103
3,5000000,3.32,159
4,10000000,5.27,245


**Conclusión:** mínimo de **1 M USD/día**. Quita solo el 1,6 % de los días, las etapas claramente inoperables, sin tocar el periodo normal de casi ninguna empresa. Los precios no se modifican: el filtro solo decide si un día puede tener señal.

## 4. ¿Cuándo hay señal de compra o de venta?

Hay tres mecanismos independientes. La venta es siempre el espejo de la compra.

| Mecanismo | Régimen | Compra | Venta |
|---|---|---|---|
| **1. Cruce de EMA** | Solo tendencia (ADX ≥ 25) | EMA(4) cruza por encima de EMA(8) y CMF ≥ +0,07 | EMA(4) cruza por debajo de EMA(8) y CMF ≤ −0,07 |
| **2. Reversión por RSI** | Solo lateral (ADX < 25) | El RSI vuelve a subir por encima de 30 | El RSI vuelve a bajar por debajo de 70 |
| **3. Ruptura de Donchian** | Cualquiera | El cierre supera el máximo de las 20 sesiones anteriores | El cierre cae bajo el mínimo de las 20 sesiones anteriores |

Hay señal si se cumple **cualquiera** de los tres, el día es líquido y ya pasó el calentamiento. Si un día tuviera compra y venta a la vez se marcaría como **conflicto**, sin operación.

Las reglas están en `detectar_senales` (`funciones.py`), para que el screener y el backtesting usen exactamente las mismas. Ejemplo con las últimas señales de Apple:

In [5]:
columnas_indicadores = list(indicadores.columns.drop(["fecha", "ticker"]))

precios_apple = datos[datos["ticker"] == "AAPL"].set_index("fecha")
senales_apple = detectar_senales(precios_apple[["close", "volume"]], precios_apple[columnas_indicadores])
senales_apple[senales_apple["senal"].notna()].tail(8)

,regimen,liquido,compra_ema,venta_ema,compra_rsi,venta_rsi,compra_donchian,venta_donchian,senal
fecha,,,,,,,,,
2026-07-16,tendencia,True,False,False,False,False,True,False,compra
2026-07-27,tendencia,True,False,False,False,False,True,False,compra
2026-07-28,tendencia,True,False,False,False,False,True,False,compra
2026-09-01,lateral,True,False,False,False,False,True,False,compra
2026-09-11,lateral,True,False,False,False,False,True,False,compra
2026-09-17,lateral,True,False,False,False,False,True,False,compra
2026-09-21,lateral,True,False,False,False,False,True,False,compra
2026-09-22,lateral,True,False,False,False,False,True,False,compra


**Conclusión:** cada fila dice qué mecanismo se activó, en qué régimen estaba el día y si era líquido.

## 5. ¿Cuántas señales salen y en qué años?

Se aplica la misma función a cada una de las 1.000 empresas.

In [6]:
inicio_calculo = time.time()
tablas = []
for ticker, datos_ticker in datos.groupby("ticker"):
    datos_ticker = datos_ticker.set_index("fecha")
    senales_ticker = detectar_senales(datos_ticker[["close", "volume"]], datos_ticker[columnas_indicadores])
    senales_ticker.insert(0, "ticker", ticker)
    tablas.append(senales_ticker.reset_index())

senales = pd.concat(tablas, ignore_index=True)
print(f"Calculado en {time.time() - inicio_calculo:.1f} s · filas: {len(senales)}")

Calculado en 2.0 s · filas: 2281091


Días en que se activa cada mecanismo, sin y con el filtro de liquidez. Un mismo día puede activar varios a la vez.

In [7]:
mecanismos = ["compra_ema", "compra_rsi", "compra_donchian", "venta_ema", "venta_rsi", "venta_donchian"]
recuento = pd.DataFrame({
    "días (sin filtro)": senales[mecanismos].sum(),
    "días (con filtro de liquidez)": senales.loc[senales["liquido"], mecanismos].sum(),
})
recuento

,días (sin filtro),días (con filtro de liquidez)
compra_ema,10525,10307
compra_rsi,4888,4759
compra_donchian,175272,173369
venta_ema,8836,8550
venta_rsi,8525,8431
venta_donchian,114842,113159


In [8]:
dias_con_senal = senales["senal"].isin(["compra", "venta"])
print(senales["senal"].value_counts().to_string())
print(f"Días con señal: {dias_con_senal.mean():.1%} de todos los empresa-día")

por_anio = senales[dias_con_senal].groupby([senales["fecha"].dt.year, "senal"]).size()
por_anio = por_anio.rename("días").reset_index().rename(columns={"fecha": "año"})
grafico = px.bar(por_anio, x="año", y="días", color="senal", barmode="group",
                 title="Días con señal por año y sentido")
grafico.show(renderer="notebook_connected")

senal
compra    187837
venta     129549
Días con señal: 13.9% de todos los empresa-día


- **Donchian domina** porque en una subida sostenida el cierre marca un máximo varios días seguidos. Solo el 64 % son rupturas nuevas, y el backtesting abre una sola posición por empresa.
- Ningún día tiene conflicto.
- **2022 y 2018**, los dos años bajistas, son los que más ventas tienen en proporción a las compras.

**Conclusión:** las reglas reaccionan al mercado como se espera. Si además aciertan lo dirá el backtesting (notebook 09).

## 6. ¿Cómo se ve sobre un gráfico? (Apple, último año)

Triángulos verdes = compra, rojos = venta. Fondo gris = régimen de tendencia.

In [9]:
desde = precios_apple.index.max() - pd.DateOffset(years=1)
p = precios_apple[precios_apple.index >= desde]
s = senales_apple[senales_apple.index >= desde]

grafico = go.Figure()
grafico.add_trace(go.Candlestick(x=p.index, open=p["open"], high=p["high"], low=p["low"], close=p["close"], name="Precio"))

compras = s.index[s["senal"] == "compra"]
ventas = s.index[s["senal"] == "venta"]
grafico.add_trace(go.Scatter(x=compras, y=p.loc[compras, "low"], mode="markers", name="compra",
                             marker={"symbol": "triangle-up", "size": 11, "color": "green"}))
grafico.add_trace(go.Scatter(x=ventas, y=p.loc[ventas, "high"], mode="markers", name="venta",
                             marker={"symbol": "triangle-down", "size": 11, "color": "red"}))

# Fondo gris en los tramos de tendencia: se recorre día a día y se anota dónde empieza y acaba cada tramo
inicio_tramo = None
dia_anterior = None
for dia, regimen in s["regimen"].items():
    if regimen == "tendencia" and inicio_tramo is None:
        inicio_tramo = dia
    if regimen != "tendencia" and inicio_tramo is not None:
        grafico.add_vrect(x0=inicio_tramo, x1=dia_anterior, fillcolor="gray", opacity=0.12, line_width=0)
        inicio_tramo = None
    dia_anterior = dia
if inicio_tramo is not None:
    grafico.add_vrect(x0=inicio_tramo, x1=dia_anterior, fillcolor="gray", opacity=0.12, line_width=0)

grafico.update_layout(title="AAPL: señales del último año (gris = tendencia)", height=550, xaxis_rangeslider_visible=False)
grafico.show(renderer="notebook_connected")

**Conclusión:** las compras aparecen sobre todo en subidas (rupturas) y en rebotes tras caídas en régimen lateral (RSI).

## 7. ¿Cuándo se cerraría una operación? (stops)

Los stops se miden en **ATR**, que se toma el día de la señal y se congela durante toda la operación.

- **Stop fijo:** entrada − 2 × ATR. Limita la pérdida si la operación sale mal desde el principio.
- **Trailing-stop:** máximo alcanzado hasta **ayer** − 3 × ATR. Sube con el precio y protege las ganancias.
- **Stop activo:** el más alto de los dos. En una venta todo es el espejo.

Ejemplo con la última compra de Apple que tiene 60 sesiones posteriores, entrando a la apertura del día siguiente, como hará el backtesting:

In [10]:
fechas_apple = senales_apple.index
compras_apple = fechas_apple[senales_apple["senal"] == "compra"]
compras_apple = compras_apple[compras_apple < fechas_apple[-61]]   # que queden 60 sesiones después
dia_senal = compras_apple[-1]
posicion_senal = fechas_apple.get_loc(dia_senal)

operacion = precios_apple.iloc[posicion_senal + 1 : posicion_senal + 61]   # 60 sesiones desde la entrada
precio_entrada = operacion["open"].iloc[0]
atr_entrada = precios_apple.loc[dia_senal, "atr"]
stops_ejemplo = calcular_stops_operacion(operacion, precio_entrada, atr_entrada, "compra")

print(f"Señal: {dia_senal.date()} · entrada: {operacion.index[0].date()} a {precio_entrada:.2f} USD · ATR: {atr_entrada:.2f}")
print(f"Stop fijo: {stops_ejemplo['stop_fijo'].iloc[0]:.2f} USD")

tocado = operacion["low"] <= stops_ejemplo["stop_activo"]
if tocado.any():
    dia_salida = tocado.idxmax()
    print(f"El precio toca el stop activo el {dia_salida.date()}")
else:
    dia_salida = None
    print("El stop no se toca en 60 sesiones: se cerraría por tiempo")

Señal: 2026-07-15 · entrada: 2026-07-16 a 327.73 USD · ATR: 8.32
Stop fijo: 311.09 USD
El precio toca el stop activo el 2026-07-31


In [11]:
grafico = go.Figure()
grafico.add_trace(go.Candlestick(x=operacion.index, open=operacion["open"], high=operacion["high"],
                                 low=operacion["low"], close=operacion["close"], name="Precio"))
grafico.add_trace(go.Scatter(x=stops_ejemplo.index, y=stops_ejemplo["stop_fijo"], name="Stop fijo (−2 ATR)", line={"dash": "dot"}))
grafico.add_trace(go.Scatter(x=stops_ejemplo.index, y=stops_ejemplo["stop_trailing"], name="Trailing (máx. − 3 ATR)", line={"dash": "dash"}))
grafico.add_trace(go.Scatter(x=stops_ejemplo.index, y=stops_ejemplo["stop_activo"], name="Stop activo", line={"width": 3, "shape": "hv"}))
grafico.add_hline(y=precio_entrada, line_width=1, annotation_text="entrada")
if dia_salida is not None:
    grafico.add_vline(x=dia_salida, line_dash="dash", line_color="red")
grafico.update_layout(title=f"AAPL: stops de la compra del {dia_senal.date()}", height=500, xaxis_rangeslider_visible=False)
grafico.show(renderer="notebook_connected")

**Conclusión:** al principio manda el stop fijo; en cuanto el precio sube más de 1 ATR, manda el trailing y va subiendo con él. El precio exacto de salida y los costes se simulan en el backtesting (notebook 09).

## 8. ¿Qué riesgo tiene cada empresa? (volatilidad y Sharpe)

- **Volatilidad anualizada:** desviación típica de los retornos diarios × √252. Un 30 % significa que en un año típico el precio se mueve en torno a ±30 %.
- **Sharpe:** retorno medio / volatilidad, anualizado, con tasa libre de riesgo del 0 %. Mide cuánto retorno hubo por cada unidad de riesgo.

Se calculan sobre el **último año**, porque junto a una señal interesa el riesgo actual.

In [12]:
filas = []
for ticker, datos_ticker in datos.groupby("ticker"):
    ultimo_anio = datos_ticker.iloc[-253:]   # 253 cierres = 252 retornos diarios
    medidas = calcular_volatilidad_y_sharpe(ultimo_anio.set_index("fecha")["close"])
    medidas["ticker"] = ticker
    medidas["atr_porcentaje"] = ultimo_anio["atr"].iloc[-1] / ultimo_anio["close"].iloc[-1]
    filas.append(medidas)

riesgo = pd.DataFrame(filas).set_index("ticker")[["volatilidad", "sharpe", "retorno_medio_anual", "atr_porcentaje", "sesiones"]]

print(f"Volatilidad mediana: {riesgo['volatilidad'].median():.0%}")
print(f"Sharpe mediano: {riesgo['sharpe'].median():.2f}")
print(f"ATR mediano: {riesgo['atr_porcentaje'].median():.1%} del precio")

Volatilidad mediana: 38%
Sharpe mediano: 0.44
ATR mediano: 3.1% del precio


In [13]:
grafico = px.scatter(riesgo.reset_index(), x="volatilidad", y="sharpe", hover_name="ticker", opacity=0.6,
                     labels={"volatilidad": "Volatilidad anualizada (último año)", "sharpe": "Sharpe (último año)"},
                     title="Riesgo y retorno ajustado por riesgo de las 1.000 empresas, último año")
grafico.add_hline(y=0, line_width=1)
grafico.update_xaxes(tickformat=".0%")
grafico.show(renderer="notebook_connected")

- La volatilidad mediana (~38 %) es muy superior a la de un índice (12–15 %): son acciones sueltas, no carteras.
- Con un ATR mediano del 3 %, el stop fijo típico queda un 6 % por debajo de la entrada.
- ¿Por qué no 10 años? Chord Energy (`CHRD`) saldría con un 8.150 % de volatilidad por una anomalía de su reestructuración de 2020. Con el último año sale un 40 %.
- Siete empresas (TECH, MKTX, AES, ACA, CZR, LNTH, PEN) tienen el precio casi congelado (ATR de entre el 0,25 % y el 0,5 % del precio), probablemente por una oferta de compra pendiente. **Pendiente decidir** si el screener y el backtesting las excluyen.

**Conclusión:** volatilidad y Sharpe son contexto para la ficha de cada empresa, no un criterio de selección: el Sharpe de un solo año es muy ruidoso.

## 9. ¿Hay meses mejores que otros? (estacionalidad)

Para cada empresa, con los **últimos 5 años**: retorno medio de cada mes del año y de cada día de la semana. Solo cuentan meses completos.

In [14]:
tablas_mes = []
tablas_dia = []
for ticker, datos_ticker in datos.groupby("ticker"):
    por_mes, por_dia = calcular_estacionalidad(datos_ticker.set_index("fecha")["close"])
    tablas_mes.append(por_mes.rename_axis("mes").reset_index().assign(ticker=ticker))
    tablas_dia.append(por_dia.rename_axis("dia_semana").reset_index().assign(ticker=ticker))

estacionalidad_mensual = pd.concat(tablas_mes, ignore_index=True)[["ticker", "mes", "retorno_medio", "porcentaje_anios_positivos", "anios"]]
estacionalidad_semanal = pd.concat(tablas_dia, ignore_index=True)[["ticker", "dia_semana", "retorno_medio", "dias"]]
print("Filas mensuales:", len(estacionalidad_mensual), "· semanales:", len(estacionalidad_semanal))

Filas mensuales: 11916 · semanales: 5000


In [15]:
media_por_mes = estacionalidad_mensual.groupby("mes", sort=False)["retorno_medio"].mean().reset_index()
grafico = px.bar(media_por_mes, x="mes", y="retorno_medio",
                 labels={"mes": "Mes", "retorno_medio": "Retorno medio del mes"},
                 title="Retorno medio por mes, media de las 1.000 empresas (últimos 5 años: 5 datos por mes)")
grafico.update_yaxes(tickformat=".1%")
grafico.show(renderer="notebook_connected")

Septiembre es el único mes claramente negativo, y julio y noviembre los mejores. Pero cada barra es la media de **solo 5 datos**: las 1.000 empresas viven los mismos 5 años, así que promediarlas no añade observaciones independientes. Por días de la semana, el lunes sale como el mejor día, lo contrario del clásico "efecto lunes". Es otra muestra de lo frágiles que son estos patrones.

**Conclusión:** la estacionalidad es solo informativa y no participa en las señales.

## 10. Guardar las salidas

In [16]:
salidas = {
    "senales.parquet": senales,
    "riesgo_empresas.parquet": riesgo.reset_index(),
    "estacionalidad_mensual.parquet": estacionalidad_mensual,
    "estacionalidad_semanal.parquet": estacionalidad_semanal,
}
for nombre, tabla in salidas.items():
    ruta = CARPETA_CACHE / nombre
    tabla.to_parquet(ruta, index=False)
    print(f"Guardado: {nombre} - {len(tabla)} filas - {ruta.stat().st_size / 1e6:.1f} MB")

Guardado: senales.parquet - 2281091 filas - 2.2 MB
Guardado: riesgo_empresas.parquet - 1000 filas - 0.0 MB
Guardado: estacionalidad_mensual.parquet - 11916 filas - 0.1 MB
Guardado: estacionalidad_semanal.parquet - 5000 filas - 0.1 MB


## Comprobaciones

In [17]:
# Una fila de señales por cada fila de datos, en el mismo orden
assert senales[["fecha", "ticker"]].equals(datos[["fecha", "ticker"]])

# Ninguna señal durante el calentamiento ni en días no líquidos
calentamiento = datos["numero_sesion"] < SESIONES_CALENTAMIENTO
assert not senales.loc[calentamiento, mecanismos].any().any()
assert senales.loc[~senales["liquido"], "senal"].isna().all()

# Cada mecanismo respeta su régimen: EMA solo en tendencia, RSI solo en lateral
assert (senales.loc[senales["compra_ema"] | senales["venta_ema"], "regimen"] == "tendencia").all()
assert (senales.loc[senales["compra_rsi"] | senales["venta_rsi"], "regimen"] == "lateral").all()

# Solo valores de señal previstos
assert set(senales["senal"].dropna().unique()) <= {"compra", "venta", "conflicto"}
print("Señales: OK")

Señales: OK


In [18]:
# Compra a 100 con ATR 2, calculada a mano
precios_prueba = pd.DataFrame({"high": [101, 104, 110, 108], "low": [99, 100, 105, 103]})
stops_prueba = calcular_stops_operacion(precios_prueba, precio_entrada=100, atr_entrada=2, sentido="compra")
assert stops_prueba["stop_fijo"].tolist() == [96, 96, 96, 96]           # 100 − 2 × 2
assert stops_prueba["stop_trailing"].tolist() == [94, 95, 98, 104]      # máximo hasta AYER − 3 × 2
assert stops_prueba["stop_activo"].tolist() == [96, 96, 98, 104]        # el más alto de los dos

# Venta a 100 con ATR 2: el espejo
precios_prueba = pd.DataFrame({"high": [101, 100, 95, 97], "low": [99, 96, 90, 92]})
stops_prueba = calcular_stops_operacion(precios_prueba, precio_entrada=100, atr_entrada=2, sentido="venta")
assert stops_prueba["stop_trailing"].tolist() == [106, 105, 102, 96]    # mínimo hasta AYER + 3 × 2
assert stops_prueba["stop_activo"].tolist() == [104, 104, 102, 96]      # el más bajo de los dos

# El stop de hoy no depende del precio de hoy: si se cambia el máximo de la sesión 10,
# los stops hasta la sesión 10 incluida no cambian (no se mira al futuro dentro del día)
precios_modificados = operacion.copy()
precios_modificados.loc[precios_modificados.index[10], "high"] *= 1.5
stops_modificados = calcular_stops_operacion(precios_modificados, precio_entrada, atr_entrada, "compra")
assert stops_modificados["stop_activo"].iloc[:11].equals(stops_ejemplo["stop_activo"].iloc[:11])
print("Stops: OK")

Stops: OK


In [19]:
# Una fila por empresa, sin vacíos
assert len(riesgo) == 1000 and not riesgo.isna().any().any()

# La volatilidad y el Sharpe de Apple coinciden con la fórmula escrita directamente
retornos_apple = precios_apple["close"].iloc[-253:].pct_change().dropna()
assert np.isclose(riesgo.loc["AAPL", "volatilidad"], retornos_apple.std() * np.sqrt(252))
assert np.isclose(riesgo.loc["AAPL", "sharpe"], retornos_apple.mean() / retornos_apple.std() * np.sqrt(252))
print("Volatilidad y Sharpe: OK")

Volatilidad y Sharpe: OK
